In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from pymatgen.io.cif import CifParser

# ================================================================
#                     CONSTANTS AND PARAMETERS
# ================================================================
a_B = 0.529  # Bohr radius in Å
pi = np.pi

# (ħ^2 / 2m_e) in eV·Å^2, so Ef(eV) = CONST * kF(Å^-1)^2
HBAR2_OVER_2ME_eVA2 = 3.80998

# ---------- Valence electrons: ONLY s + p orbitals ----------
# Count = electrons in the OUTERMOST s subshell of the neutral ground-state atom.
# Anomalous ns^1 metals (Cr, Nb, Mo, Ru, Rh, Pt, Au) therefore give 1; Pd (4d^10 5s^0) gives 0.
VALENCE_SP = {
    "Sc": 2, "Y": 2,
    "Ti": 2, "Zr": 2, "Hf": 2,
    "V": 2, "Nb": 1, "Ta": 2,
    "Cr": 1, "Mo": 1, "W": 2,
    "Mn": 2, "Tc": 2, "Re": 2,
    "Fe": 2, "Ru": 1, "Os": 2,
    "Co": 2, "Rh": 1, "Ir": 2,
    "Ni": 2, "Pd": 0, "Pt": 1,
    "Zn": 2, "Cd": 2, "Hg": 2,
    "Cl": 7, "Br": 7, "I": 7,
}

B_CONST = 2.25
HALIDES = ["Cl", "Br", "I"]

# ================================================================
#                     HELPER FUNCTIONS
# ================================================================
def _to_float(x):
    """Parse CIF numeric fields robustly (handles lists, uncertainties like 12.3(4))."""
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        x = x[0] if len(x) else None
    if x is None:
        return None
    s = str(x).strip()
    if s in ("?", ".", ""):
        return None
    s = re.sub(r"\(.*\)$", "", s)  # 12.34(5) -> 12.34
    try:
        return float(s)
    except Exception:
        return None


def _to_int(x):
    f = _to_float(x)
    return None if f is None else int(round(f))


def _to_str(x):
    """Parse CIF string fields robustly (handles lists, quotes, ?, .)."""
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        x = x[0] if len(x) else None
    if x is None:
        return None
    s = str(x).strip()
    if s in ("?", ".", ""):
        return None
    # remove surrounding quotes if present
    if (s.startswith("'") and s.endswith("'")) or (s.startswith('"') and s.endswith('"')):
        s = s[1:-1].strip()
    return s if s else None


def read_best_cif_block(cif_path):
    """
    Choose the best CIF data block that contains numeric Z.
    Prefer blocks with atom coordinates and volume.
    """
    parser = CifParser(cif_path)
    d = parser.as_dict()

    best = None
    for name, block in d.items():
        Z = _to_int(block.get("_cell_formula_units_Z"))
        if Z is None:
            continue

        V = _to_float(block.get("_cell_volume"))
        has_atoms = (
            "_atom_site_fract_x" in block
            and block.get("_atom_site_fract_x") not in (None, "?", ".", ["?"], ["."])
        )

        score = (2 if has_atoms else 0) + (1 if V is not None else 0)

        if best is None or score > best[0]:
            best = (score, name, block)

    if best is None:
        raise ValueError("Could not find CIF data block with numeric _cell_formula_units_Z")
    return best[2]


def read_Z_Vcell_sg_from_cif(cif_path):
    """
    Read Z, Vcell, and symmetry space-group (name + number) directly from CIF.
    If _cell_volume is missing, compute from a,b,c,alpha,beta,gamma.
    Returns: (Z_int, Vcell_float, sg_name_str, sg_number_int)
    """
    block = read_best_cif_block(cif_path)

    Z = _to_int(block.get("_cell_formula_units_Z"))
    if Z is None:
        raise ValueError("CIF has no usable _cell_formula_units_Z")

    V = _to_float(block.get("_cell_volume"))
    if V is None:
        a = _to_float(block.get("_cell_length_a"))
        b = _to_float(block.get("_cell_length_b"))
        c = _to_float(block.get("_cell_length_c"))
        alpha = _to_float(block.get("_cell_angle_alpha"))
        beta = _to_float(block.get("_cell_angle_beta"))
        gamma = _to_float(block.get("_cell_angle_gamma"))

        if None in (a, b, c, alpha, beta, gamma):
            raise ValueError("CIF missing _cell_volume and insufficient lattice parameters")

        ar, br, gr = np.deg2rad([alpha, beta, gamma])
        V = a * b * c * np.sqrt(
            1
            + 2 * np.cos(ar) * np.cos(br) * np.cos(gr)
            - np.cos(ar) ** 2
            - np.cos(br) ** 2
            - np.cos(gr) ** 2
        )

    # --- Space group: read directly from CIF block (support common CIF tags) ---
    sg_name = (
        _to_str(block.get("_symmetry_space_group_name_H-M"))
        or _to_str(block.get("_symmetry_space_group_name_H-M_alt"))
        or _to_str(block.get("_space_group_name_H-M_alt"))
        or _to_str(block.get("_space_group_name_H-M_ref"))
        or _to_str(block.get("_space_group_name_H-M"))
    )

    sg_number = (
        _to_int(block.get("_symmetry_Int_Tables_number"))
        or _to_int(block.get("_space_group_IT_number"))
        or _to_int(block.get("_space_group_IT_number"))
    )

    return Z, float(V), sg_name, sg_number


def read_cif_reliably(cif_path):
    """Parse CIF safely, return a valid Structure (used for bonding & element IDs)."""
    parser = CifParser(cif_path)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning)
        structs = parser.parse_structures(primitive=False)
    for s in structs:
        if s and len(s) > 0:
            return s
    raise ValueError("No valid structure in CIF")


def site_symbol(site):
    """Robust element symbol getter for pymatgen PeriodicSite (ordered/disordered)."""
    if hasattr(site, "specie") and site.specie is not None:
        return site.specie.symbol
    if hasattr(site, "species") and site.species is not None:
        sp = max(site.species.items(), key=lambda kv: kv[1])[0]
        return sp.symbol
    return str(site).split()[0]


def identify_metal_halide(structure):
    """Identify metal (M) and halide (X), ignoring common junk species if present."""
    elems = [el.symbol for el in structure.composition.elements]
    halides = [e for e in elems if e in HALIDES]
    non_halides = [e for e in elems if e not in HALIDES]

    junk = {"H", "C", "N", "O"}
    metal = next((e for e in non_halides if e not in junk), None) or (non_halides[0] if non_halides else None)
    halide = halides[0] if halides else None
    return metal, halide


def get_mean_mx_bond_length(structure, metal, halide, r_cut=3.6):
    """Compute average M–X bond length (robust for disordered sites)."""
    bonds = []
    for site in structure:
        if site_symbol(site) == metal:
            for nbor in structure.get_neighbors(site, r_cut):
                if site_symbol(nbor) == halide:
                    bonds.append(nbor.nn_distance)
    return float(np.mean(bonds)) if bonds else 2.6


def parse_stoichiometry(name):
    """Extract m, n from formula (e.g. FeBr2)."""
    base = re.sub(r"\s+(1T'?|2H|3R)\s*$", "", name.strip())
    parts = re.findall(r"([A-Z][a-z]*)(\d*)", base)
    m, n_atoms = 1, 2
    if len(parts) == 2:
        m = int(parts[0][1]) if parts[0][1] else 1
        n_atoms = int(parts[1][1]) if parts[1][1] else 1
    return m, n_atoms


def compute_ks_from_structure(Vcell, cell_formula_units_Z, valence_sp_per_fu):
    """Electron density: n = (valence_sp_per_fu × Z) / Vcell"""
    n = float(valence_sp_per_fu) * float(cell_formula_units_Z) / float(Vcell)
    kF = (3 * pi**2 * n) ** (1.0 / 3.0)
    ks = np.sqrt(4.0 * kF / (pi * a_B))
    return ks, n, kF


# ================================================================
#                     MAIN EXECUTION
# ================================================================
excel_in = "Bandgaps-Layered TMH-Pollini-input.xlsx"
cif_dir = "CIFs of Layered TMHs"
excel_out = "Bandgaps-Layered TMH-Pollini-output.xlsx"

df = pd.read_excel(excel_in)
rows = []

for i, row in df.iterrows():
    material = str(row["Material"]).strip()
    cif_path = os.path.join(cif_dir, f"{material}-SM.cif")
    print(f"\n[{i+1}/{len(df)}] {material}")

    if not os.path.exists(cif_path):
        print("   ⚠ CIF not found")
        continue

    try:
        # --- Read Z, Vcell, and space-group info from CIF directly (authoritative) ---
        cell_formula_units_Z, Vcell, sg_name, sg_number = read_Z_Vcell_sg_from_cif(cif_path)

        # --- Read structure only for bonding + element identification ---
        s = read_cif_reliably(cif_path)

        metal, halide = identify_metal_halide(s)
        if metal is None or halide is None:
            raise ValueError("Could not identify metal/halide from structure")

        m, n_atoms = parse_stoichiometry(material)
        if metal not in VALENCE_SP:
            raise KeyError(f"{material}: metal '{metal}' not in VALENCE_SP — add it explicitly.")
        ZA_sp = VALENCE_SP[metal]
        ZB_sp = VALENCE_SP[halide]
        valence_sp_per_fu = ZA_sp * m + ZB_sp * n_atoms

        d = get_mean_mx_bond_length(s, metal, halide, r_cut=3.6)
        r0 = d / 2.0

        ks, n, kF = compute_ks_from_structure(Vcell, cell_formula_units_Z, valence_sp_per_fu)
        Ef = HBAR2_OVER_2ME_eVA2 * (kF ** 2)

        Egc = 40.5 / (d ** 2.5)
        term_abs = abs(ZA_sp - ZB_sp) / r0
        Egi = 14.4 * B_CONST * np.exp(-ks * r0) * term_abs
        Eg = np.sqrt(Egc**2 + Egi**2)
        fi = Egi**2 / Eg**2 

        rows.append({
            "Material": material,
            "symmetry_space_group_name_H-M": sg_name,
            "symmetry_Int_Tables_number": sg_number,
            "cell_formula_units_Z": cell_formula_units_Z,
            "valence_sp_per_fu": valence_sp_per_fu,
            "d (Å)": d,
            "r0 (Å)": r0,
            "Vcell (Å^3)": Vcell,
            "n (Å^-3)": n,
            "kF (Å^-1)": kF,
            "Ks (Å^-1)": ks,
            "Ef (eV)": Ef,
            "Egc (eV)": Egc,
            "Egi (eV)": Egi,
            "Eg (eV)": Eg,
            "fi": fi, 
        })

        print(
            f"   ✅ Z={cell_formula_units_Z}, Vcell={Vcell:.3f} Å³, "
            f"SG={sg_name} ({sg_number}), "
            f"n={n:.4e} Å⁻³, kF={kF:.3f} Å⁻¹, Ef={Ef:.2f} eV, Eg={Eg:.2f} eV"
        )

    except Exception as e:
        print(f"   ❌ Error: {e}")

pd.DataFrame(rows).to_excel(excel_out, index=False)
print(f"\n✅ Done. Results saved as '{excel_out}'")


[1/29] CdCl2
   ✅ Z=3, Vcell=225.290 Å³, SG=R-3m (166), n=2.1306e-01 Å⁻³, kF=1.848 Å⁻¹, Ef=13.01 eV, Eg=8.15 eV

[2/29] CdBr2
   ✅ Z=3, Vcell=259.100 Å³, SG=R-3m (166), n=1.8526e-01 Å⁻³, kF=1.764 Å⁻¹, Ef=11.85 eV, Eg=7.30 eV

[3/29] CdI2
   ✅ Z=1, Vcell=106.400 Å³, SG=P-3m1 (164), n=1.5038e-01 Å⁻³, kF=1.645 Å⁻¹, Ef=10.31 eV, Eg=6.16 eV

[4/29] CoCl2
   ✅ Z=3, Vcell=190.100 Å³, SG=R-3m (166), n=2.5250e-01 Å⁻³, kF=1.955 Å⁻¹, Ef=14.57 eV, Eg=9.38 eV

[5/29] CoBr2
   ✅ Z=1, Vcell=72.000 Å³, SG=P-3m1 (164), n=2.2222e-01 Å⁻³, kF=1.874 Å⁻¹, Ef=13.38 eV, Eg=8.47 eV

[6/29] CoI2
   ✅ Z=1, Vcell=92.700 Å³, SG=P-3m1 (164), n=1.7260e-01 Å⁻³, kF=1.722 Å⁻¹, Ef=11.30 eV, Eg=6.90 eV

[7/29] CrCl3
   ✅ Z=4, Vcell=356.570 Å³, SG=C 1 2/m 1 (12), n=2.4680e-01 Å⁻³, kF=1.941 Å⁻¹, Ef=14.35 eV, Eg=14.04 eV

[8/29] CrBr3
   ✅ Z=6, Vcell=634.300 Å³, SG=R-3 (148), n=2.0810e-01 Å⁻³, kF=1.833 Å⁻¹, Ef=12.81 eV, Eg=9.89 eV

[9/29] CrI3
   ✅ Z=4, Vcell=540.480 Å³, SG=C 1 2/m 1 (12), n=1.6282e-01 Å⁻³, kF=1.689 Å⁻¹, E